# CERPT 1B 한국어 30 epoch 학습

위에서 아래 순서로 실행합니다. 데이터, tokenizer, checkpoint, 최종 모델은 Google Drive에 저장됩니다. 런타임이 끊어지면 Drive를 다시 마운트하고 같은 셀들을 실행하면 마지막 `checkpoint-*`에서 자동으로 이어집니다.

무료 Colab GPU와 세션 시간은 보장되지 않습니다. 먼저 **런타임 > 런타임 유형 변경 > GPU**를 선택하세요.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("GPU 런타임이 아닙니다. 런타임 유형을 GPU로 변경하세요.")
gpu = torch.cuda.get_device_properties(0)
gpu_memory_gib = gpu.total_memory / 1024**3
print({"gpu": gpu.name, "memory_gib": round(gpu_memory_gib, 2), "torch": torch.__version__})
if gpu_memory_gib < 14.5:
    raise RuntimeError("CERPT 1B 학습에는 최소 14.5 GiB GPU 메모리가 필요합니다.")

## 1. Drive와 사용자 설정

오피스 ZIP의 학습 및 파생 모델 공개 조건을 직접 확인한 뒤 `ACKNOWLEDGE_OFFICE_LICENSE = True`로 바꾸세요. ZIP이 지정 경로에 없으면 업로드 창이 열리고 Drive에 복사됩니다.

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

REPO_URL = "https://github.com/dkekzhs/cerpt-model.git"
REPO_BRANCH = "main"
PROJECT_ROOT = Path("/content/cerpt-model")
CERPT_CLOUD_ROOT = Path("/content/drive/MyDrive/cerpt-cloud")
OFFICE_ZIP = Path("/content/drive/MyDrive/대화데이터_오피스(JSON).zip")
HF_REPO_ID = "qweqwqw113/cerpt-causal-korean-v7-1b-30ep"
ACKNOWLEDGE_OFFICE_LICENSE = False

CERPT_CLOUD_ROOT.mkdir(parents=True, exist_ok=True)

In [ ]:
from google.colab import files

if not ACKNOWLEDGE_OFFICE_LICENSE:
    raise RuntimeError("오피스 데이터의 이용 조건을 확인한 뒤 acknowledgement를 True로 바꾸세요.")
if not OFFICE_ZIP.is_file():
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise RuntimeError("오피스 ZIP 파일 하나만 업로드하세요.")
    uploaded_name = next(iter(uploaded))
    OFFICE_ZIP.write_bytes(uploaded[uploaded_name])
print(OFFICE_ZIP)

## 2. 코드와 실행 환경 준비

Colab에 이미 설치된 CUDA PyTorch는 유지하고, CERPT가 검증한 나머지 패키지만 설치합니다.

In [ ]:
import os
import subprocess

if (PROJECT_ROOT / ".git").is_dir():
    subprocess.run(["git", "-C", str(PROJECT_ROOT), "fetch", "origin", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", str(PROJECT_ROOT), "switch", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", str(PROJECT_ROOT), "pull", "--ff-only", "origin", REPO_BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(PROJECT_ROOT)], check=True)
os.chdir(PROJECT_ROOT)
required_paths = (
    PROJECT_ROOT / "scripts/prepare_korean_conversations.py",
    PROJECT_ROOT / "scripts/train_korean_tokenizer.py",
    PROJECT_ROOT / "scripts/train_causal_cloud.py",
)
missing_paths = [str(path.relative_to(PROJECT_ROOT)) for path in required_paths if not path.is_file()]
if missing_paths:
    raise RuntimeError(f"Git checkout에 필수 학습 파일이 없습니다: {missing_paths}")
print(subprocess.run(["git", "rev-parse", "--short", "HEAD"], check=True, capture_output=True, text=True).stdout.strip())

In [ ]:
import shutil
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
uv = shutil.which("uv")
if not uv:
    raise RuntimeError("uv 실행 파일을 찾을 수 없습니다.")
runtime_dependencies = [
    "accelerate==1.14.0",
    "huggingface_hub==1.24.0",
    "pydantic==2.11.4",
    "tokenizers==0.22.2",
    "transformers==5.7.0",
]
subprocess.run([uv, "pip", "install", "--system", "--python", sys.executable, *runtime_dependencies], check=True)
subprocess.run([uv, "pip", "install", "--system", "--python", sys.executable, "--no-deps", "-e", str(PROJECT_ROOT)], check=True)
training_arguments_probe = "; ".join([
    "import inspect, transformers",
    "from transformers import TrainingArguments",
    "print({'transformers': transformers.__version__, 'module': TrainingArguments.__module__, 'signature': str(inspect.signature(TrainingArguments.__init__))})",
    "TrainingArguments(output_dir='/tmp/cerpt-training-args-probe', warmup_steps=0.03, eval_strategy='no')",
])
subprocess.run([sys.executable, "-c", training_arguments_probe], check=True)

## 3. 세 데이터 소스 통합과 32k tokenizer

완료된 데이터와 tokenizer가 Drive에 있으면 다시 만들지 않습니다. 처음 실행할 때만 Songys 데이터를 고정 commit에서 받고 오피스 ZIP과 기존 데이터를 합칩니다.

In [ ]:
RAW_DIR = CERPT_CLOUD_ROOT / "raw"
DATA_DIR = CERPT_CLOUD_ROOT / "data/korean_conversations_v7"
TOKENIZER_DIR = CERPT_CLOUD_ROOT / "tokenizers/cerpt-korean-32k"
MODEL_DIR = CERPT_CLOUD_ROOT / "models/cerpt-causal-korean-v7-1b-30ep"
SONGYS_SHA = "4cf20d13fc46f5037fd1c531cd566e2dd9f72974"
RAW_DIR.mkdir(parents=True, exist_ok=True)

songys_csv = RAW_DIR / "ChatbotData.csv"
songys_license = RAW_DIR / "ChatbotData.LICENSE"
if not songys_csv.is_file():
    subprocess.run(["curl", "-fL", f"https://raw.githubusercontent.com/songys/Chatbot_data/{SONGYS_SHA}/ChatbotData.csv", "-o", str(songys_csv)], check=True)
if not songys_license.is_file():
    subprocess.run(["curl", "-fL", f"https://raw.githubusercontent.com/songys/Chatbot_data/{SONGYS_SHA}/LICENSE", "-o", str(songys_license)], check=True)

if not (DATA_DIR / "metadata.json").is_file():
    subprocess.run([
        sys.executable, "scripts/prepare_korean_conversations.py",
        "--current-data-dir", "data/korean_basic_v6",
        "--songys-csv", str(songys_csv),
        "--office-zip", str(OFFICE_ZIP),
        "--output-dir", str(DATA_DIR),
        "--acknowledge-office-license",
    ], cwd=PROJECT_ROOT, check=True)

In [ ]:
if not (TOKENIZER_DIR / "tokenizer.json").is_file():
    subprocess.run([
        sys.executable, "scripts/train_korean_tokenizer.py",
        "--data-dir", str(DATA_DIR),
        "--output-dir", str(TOKENIZER_DIR),
    ], cwd=PROJECT_ROOT, check=True)

In [ ]:
import json
from transformers import PreTrainedTokenizerFast

sys.path.insert(0, str(PROJECT_ROOT / "src"))
from cerpt.data.causal import add_workspace_tokens

metadata = json.loads((DATA_DIR / "metadata.json").read_text(encoding="utf-8"))
tokenizer = PreTrainedTokenizerFast.from_pretrained(TOKENIZER_DIR)
base_vocab = len(tokenizer)
workspace_ids = add_workspace_tokens(tokenizer, num_cycles=6, workspace_slots=16)
if len(tokenizer) != 32768 or len(workspace_ids) != 96:
    raise RuntimeError("tokenizer가 32,768-token architecture 계약과 맞지 않습니다.")
print({"counts": metadata["counts"], "base_vocab": base_vocab, "workspace_tokens": len(workspace_ids), "final_vocab": len(tokenizer)})

## 4. 현재 상태 확인과 30 epoch 학습

아래 학습 셀은 가장 최근 checkpoint를 자동 탐색합니다. Colab이 끊어지면 이 노트북을 다시 열고 위 셀부터 실행한 다음 같은 학습 셀을 실행하세요. Drive 용량을 아끼기 위해 Colab profile은 checkpoint 한 개만 유지합니다.

In [ ]:
completion_marker = MODEL_DIR / "TRAINING_COMPLETE"
checkpoints = sorted(MODEL_DIR.glob("checkpoint-*"), key=lambda path: int(path.name.split("-")[-1])) if MODEL_DIR.is_dir() else []
print({"complete": completion_marker.is_file(), "latest_checkpoint": str(checkpoints[-1]) if checkpoints else None})

In [ ]:
from collections import deque

training_env = {
    **os.environ,
    "PYTHONUNBUFFERED": "1",
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
}
TRAINING_LOG = MODEL_DIR / "training.log"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
training_command = [
    sys.executable, "-u", "scripts/train_causal_cloud.py",
    "--data-dir", str(DATA_DIR),
    "--tokenizer-dir", str(TOKENIZER_DIR),
    "--output-dir", str(MODEL_DIR),
    "--architecture-config", "configs/cerpt-causal-1b.json",
    "--profile", "colab-t4",
    "--epochs", "30",
    "--batch-size", "1",
    "--gradient-accumulation-steps", "32",
    "--save-steps", "100",
]
print({"training_log": str(TRAINING_LOG), "command": training_command})
recent_output = deque(maxlen=80)
with TRAINING_LOG.open("a", encoding="utf-8") as training_log:
    training_process = subprocess.Popen(
        training_command,
        cwd=PROJECT_ROOT,
        env=training_env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
        bufsize=1,
    )
    assert training_process.stdout is not None
    for line in training_process.stdout:
        print(line, end="")
        training_log.write(line)
        training_log.flush()
        recent_output.append(line)
    return_code = training_process.wait()
if return_code != 0:
    raise RuntimeError(
        f"학습 프로세스가 exit code {return_code}로 종료됐습니다. 전체 로그: {TRAINING_LOG}\n"
        + "".join(recent_output)
    )

In [ ]:
if not completion_marker.is_file():
    raise RuntimeError("학습이 아직 완료되지 않았습니다. 같은 학습 셀을 다시 실행하세요.")
print(completion_marker.read_text(encoding="utf-8"))
print(MODEL_DIR / "final")

## 5. 선택 사항: Hugging Face 업로드

Colab 왼쪽 열쇠 아이콘의 Secrets에 write 권한이 있는 `HF_TOKEN`을 추가한 뒤 실행합니다. `TRAINING_COMPLETE`가 없으면 업로드하지 않습니다. 원본 학습 데이터는 업로드하지 않습니다.

In [ ]:
from google.colab import userdata

if not completion_marker.is_file():
    raise RuntimeError("30 epoch 완료 marker가 없어 업로드를 중단합니다.")
hf_token = userdata.get("HF_TOKEN")
upload_env = {**os.environ, "HF_TOKEN": hf_token}
upload_commands = [
    ["hf", "upload", HF_REPO_ID, str(MODEL_DIR / "final"), ".", "--repo-type", "model"],
    ["hf", "upload", HF_REPO_ID, "docs/model-cards/MODEL_CARD_CAUSAL_KOREAN_1B_30EP.md", "README.md", "--repo-type", "model"],
    ["hf", "upload", HF_REPO_ID, str(completion_marker), "TRAINING_COMPLETE", "--repo-type", "model"],
    ["hf", "upload", HF_REPO_ID, str(MODEL_DIR / "trainer_state.json"), "trainer_state.json", "--repo-type", "model"],
]
for command in upload_commands:
    subprocess.run(command, cwd=PROJECT_ROOT, env=upload_env, check=True)